# B06 — OptionParser Quality Eval

Two parts:
1. **Offline classifier checks** (no API) — assert the deterministic role classifier on real
   audited option strings, especially the YNU-vs-full-claim distinction.
2. **Live structural eval** (via `/qparser`) over all 359 MCQs — extraction recall (incl. the
   13 `A)` records), no-marker-leak, option-count, role distribution, handoff correctness, % supported.


## Part 1 — Offline deterministic classifier checks (no API)
Runs against the local source; needs `PYTHONPATH=src` (kernel started in repo root).

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'src/exact').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from exact.type1.parser.oparser import classify, recover_subject, realize_fragment
from exact.type1.parser.options import extract_mcq

CASES = {
    'No one is qualified.': 'FULL_CLAIM',
    'No AI models use deep learning.': 'FULL_CLAIM',
    'No, only some earn A+.': 'YNU_ANSWER',
    'Yes, all mastered the subject.': 'YNU_ANSWER',
    'Uncertain.': 'YNU_ANSWER',
    'Cannot be determined': 'YNU_ANSWER',
    'Premises 1, 3, 7': 'PREMISE_REFERENCE',
    'None of the above can be concluded.': 'NONE_OF_ABOVE',
    '\u2200x (R(x) \u2192 P(x))': 'RAW_FOL',
    'Exists(x, Enrolled(x))': 'RAW_FOL',
    'Can be a research mentor': 'SUBJECTLESS_MODAL_FRAGMENT',
    'Cannot teach graduate courses': 'SUBJECTLESS_MODAL_FRAGMENT',
    'Eligible for the internship program': 'PREDICATE_FRAGMENT',
    'Spaced repetition improves both memory and academic performance': 'CONJUNCTIVE_CLAIM',
    'If a Python project is not optimized, then it is not well-tested': 'FULL_CLAIM',
    'Both A and B': 'UNKNOWN',
}
fails = [(t, classify(t), w) for t, w in CASES.items() if classify(t) != w]
for t, got, want in fails:
    print(f'  FAIL {got} (want {want}) <- {t!r}')
print(f'classifier: {len(CASES)-len(fails)}/{len(CASES)} correct')
assert not fails, 'deterministic classifier mismatches'

# realization preserves modality + negation
assert recover_subject('Which statement is true about Professor Kim?') == 'Professor Kim'
assert realize_fragment('SUBJECTLESS_MODAL_FRAGMENT', 'Cannot teach graduate courses', 'Professor Kim') \
    == 'Professor Kim cannot teach graduate courses.'
# extractor handles A) and A.
assert extract_mcq('Q?\nA) one\nB) two\nC) three\nD) four').marker_style == 'paren'
assert extract_mcq('Q?\nA. one\nB. two\nC. three').option_count == 3
print('realization + extractor checks passed')


## Part 2 — Live structural eval via `/qparser`

In [ ]:
import json, re, time, asyncio
from collections import Counter
import httpx

API_BASE    = "https://api.iamphuckhang.dev"
QPARSER_URL = f"{API_BASE}/qparser"
N_SAMPLES   = 20      # MCQs to eval (None = all 359)
CONCURRENCY = 8
TIMEOUT     = 120.0

DATA = ROOT / "src/exact/datasets/exact"
raw = json.load(open(DATA / "Logic_Based_Educational_Queries.json"))
_OPTION_LINE = re.compile(r"^\s*([A-E])([.)])")

def mcq_markers(q):
    return {m.group(2) for line in q.split("\n") if (m := _OPTION_LINE.match(line))}

mcqs = []
for g_idx, group in enumerate(raw):
    for q_idx, question in enumerate(group["questions"]):
        markers = mcq_markers(question)
        if markers:
            mcqs.append({
                "id": f"logic_{g_idx:04d}_{q_idx:02d}",
                "premises": group["premises-NL"],
                "question": question,
                "surface_marker": "paren" if ")" in markers else "dot",
            })
print(f"{len(mcqs)} MCQs found; A) records: {sum(1 for m in mcqs if m['surface_marker']=='paren')}")


In [ ]:
async def call(client, sem, inst):
    async with sem:
        t0 = time.perf_counter()
        err, body = None, None
        try:
            r = await client.post(QPARSER_URL,
                                  json={"question": inst["question"], "premises": inst["premises"]},
                                  timeout=TIMEOUT)
            r.raise_for_status()
            body = r.json()
        except Exception as e:
            err = repr(e)
    return {**inst, "spec": body, "latency": time.perf_counter() - t0, "error": err}

async def run_eval(items):
    sem = asyncio.Semaphore(CONCURRENCY); done = 0
    async with httpx.AsyncClient() as client:
        async def wrapped(i):
            nonlocal done
            res = await call(client, sem, i); done += 1
            if done % 5 == 0 or done == len(items): print(f'  {done}/{len(items)}', end='\r')
            return res
        return await asyncio.gather(*(wrapped(i) for i in items))

subset = mcqs[:N_SAMPLES] if N_SAMPLES else mcqs
print(f"Evaluating {len(subset)} MCQs...")
results = await run_eval(subset)
success = [r for r in results if not r["error"]]
print(f"\nSuccess: {len(success)}/{len(results)}")


In [ ]:
# --- Structural metrics ---
n = len(success)
extraction_ok = sum(1 for r in success if r["spec"]["option_claims"])
paren = [r for r in success if r["surface_marker"] == "paren"]
paren_ok = sum(1 for r in paren if r["spec"]["option_claims"])
leak = sum(1 for r in success for c in r["spec"]["option_claims"]
           if re.match(r"^\s*[A-E][.)]\s", c["normalized_text"]))
counts = Counter(len(r["spec"]["option_claims"]) for r in success)
supported = sum(1 for r in success if r["spec"]["supported"])

print("=== Extraction ===")
print(f"  options present : {extraction_ok}/{n}")
print(f"  A) coverage     : {paren_ok}/{len(paren)}")
print(f"  marker leaks    : {leak}")
print(f"  option counts   : {dict(counts)}")
print(f"  supported       : {supported}/{n}  ({supported/n:.1%})")

# --- Role distribution ---
roles = Counter(c["role"] for r in success for c in r["spec"]["option_claims"])
print("\n=== Role distribution ===")
for k, v in roles.most_common():
    print(f"  {k:28s}: {v}")

# --- Handoff correctness: special roles must NOT carry FOL / be selectable ---
SPECIAL = {"RAW_FOL", "PREMISE_REFERENCE", "YNU_ANSWER", "NONE_OF_ABOVE", "UNKNOWN"}
bad_handoff = [
    (r["id"], c["label"], c["role"])
    for r in success for c in r["spec"]["option_claims"]
    if c["role"] in SPECIAL and (c["fol"] is not None or c["is_selectable"])
]
print(f"\n=== Handoff correctness ===\n  special-role leaks to solver: {len(bad_handoff)}")
for b in bad_handoff[:10]:
    print("   ", b)


In [ ]:
# --- Spot-check table for manual role review ---
for r in success[:8]:
    s = r["spec"]
    print(f"\n{r['id']}  {s['question_format']}/{s['solver_mode']}  supported={s['supported']}")
    for c in s["option_claims"]:
        detail = c["claim_text"] or c["raw_fol"] or (c["ynu_value"] if c["ynu_value"] != "none" else c["normalized_text"][:60])
        print(f"    {c['label']}. [{c['role']:26s} sel={int(c['is_selectable'])}] {detail}")


## Notes
- Part 1 needs no server; Part 2 needs the VM/API up.
- Acceptance: options present for all MCQs, A) coverage = 100%, 0 marker leaks, counts ⊆ {3,4},
  0 special-role leaks to the solver.
- `role` taxonomy: FULL_CLAIM / CONJUNCTIVE_CLAIM / PREDICATE_FRAGMENT / SUBJECTLESS_MODAL_FRAGMENT /
  RAW_FOL / PREMISE_REFERENCE / YNU_ANSWER / NONE_OF_ABOVE / UNKNOWN.
- Set `N_SAMPLES = None` for the full 359.
